# SentinelIQ — 05 Fusion Ensemble
Combine IsolationForest, Autoencoder, and BERT scores into a unified anomaly score.

In [ ]:
import os, sys
repo = '/kaggle/working/SentinelIQ'
if os.path.exists(repo):
    !git -C /kaggle/working/SentinelIQ pull
else:
    !git clone https://github.com/hasan-rajab/SentinelIQ.git
%cd /kaggle/working/SentinelIQ
sys.path.insert(0, '/kaggle/working/SentinelIQ')
!pip install pyyaml joblib scikit-learn torch transformers shap -q


In [ ]:
import json
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay

from ml.models.isolation_forest import SentinelIsolationForest
from ml.models.autoencoder import SentinelAutoencoder
from ml.models.bert_log import SentinelBertLog
from ml.fusion.ensemble import SentinelEnsemble

sns.set_theme(style="darkgrid")

with open('configs/model_config.yaml') as f:
    cfg = yaml.safe_load(f)

def load_jsonl(path):
    return pd.DataFrame([json.loads(l) for l in open(path)])


In [ ]:
# Generate fresh data
!python data/simulated/pipeline.py --duration 120 --anomaly-rate 0.08


## Load Saved Models

In [ ]:
feature_cols = cfg['isolation_forest']['features']
net_features = cfg['network']['features']

if_metrics = SentinelIsolationForest.load('ml/saved_models', name='isolation_forest_metrics')
if_network = SentinelIsolationForest.load('ml/saved_models', name='isolation_forest_network')
ae         = SentinelAutoencoder.load('ml/saved_models', name='autoencoder_metrics')
bert       = SentinelBertLog.load('ml/saved_models', name='bert_log')
print("All models loaded.")


## Score Each Modality

In [ ]:
metrics_df = load_jsonl('data/simulated/metrics.jsonl')
logs_df    = load_jsonl('data/simulated/logs.jsonl')
network_df = load_jsonl('data/simulated/network.jsonl')

# Use metrics for IF + AE, logs for BERT
# Align on shared index using metrics as base
n = min(len(metrics_df), len(logs_df))
metrics_df = metrics_df.iloc[:n].reset_index(drop=True)
logs_df    = logs_df.iloc[:n].reset_index(drop=True)

y_true = metrics_df['is_anomaly'].astype(int).values

if_scores   = if_metrics.score(metrics_df)
ae_scores   = ae.score(metrics_df)
bert_scores = bert.score(logs_df)

print(f"IF score range   : {if_scores.min():.4f} – {if_scores.max():.4f}")
print(f"AE score range   : {ae_scores.min():.6f} – {ae_scores.max():.6f}")
print(f"BERT score range : {bert_scores.min():.4f} – {bert_scores.max():.4f}")


## Fuse Scores

In [ ]:
ensemble = SentinelEnsemble(
    weights={"isolation_forest": 0.30, "autoencoder": 0.30, "bert_log": 0.40},
    strategy="weighted_avg",
    threshold=0.5,
)

fused = ensemble.fuse(if_scores=if_scores, ae_scores=ae_scores, bert_scores=bert_scores)
print(f"Fused score range: {fused.min():.4f} – {fused.max():.4f}")


## Evaluate Ensemble vs Individual Models

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

def quick_eval(name, scores, y_true, threshold=0.5):
    from sklearn.preprocessing import MinMaxScaler
    scores_norm = MinMaxScaler().fit_transform(scores.reshape(-1,1)).flatten()
    y_pred = (scores_norm >= threshold).astype(int)
    return {
        "Model": name,
        "ROC-AUC": round(roc_auc_score(y_true, scores_norm), 4),
        "Avg Precision": round(average_precision_score(y_true, scores_norm), 4),
        "F1": round(f1_score(y_true, y_pred, zero_division=0), 4),
    }

results = pd.DataFrame([
    quick_eval("IsolationForest", if_scores, y_true),
    quick_eval("Autoencoder",     ae_scores, y_true),
    quick_eval("BERT",            bert_scores, y_true),
    {"Model": "Ensemble", "ROC-AUC": round(roc_auc_score(y_true, fused), 4),
     "Avg Precision": round(average_precision_score(y_true, fused), 4),
     "F1": round(f1_score(y_true, (fused>=0.5).astype(int), zero_division=0), 4)},
])
print(results.to_string(index=False))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC curves for all models
from sklearn.metrics import roc_curve
for name, scores in [("IF", if_scores), ("AE", ae_scores), ("BERT", bert_scores), ("Ensemble", fused)]:
    from sklearn.preprocessing import MinMaxScaler
    s = MinMaxScaler().fit_transform(scores.reshape(-1,1)).flatten()
    fpr, tpr, _ = roc_curve(y_true, s)
    auc = roc_auc_score(y_true, s)
    axes[0].plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})")
axes[0].plot([0,1],[0,1],'k--')
axes[0].set_title("ROC Curves — All Models vs Ensemble")
axes[0].set_xlabel("FPR")
axes[0].set_ylabel("TPR")
axes[0].legend()

# Fused score distribution
axes[1].hist(fused[y_true==0], bins=30, alpha=0.6, color='#2ecc71', label='Normal')
axes[1].hist(fused[y_true==1], bins=30, alpha=0.6, color='#e74c3c', label='Anomaly')
axes[1].axvline(0.5, color='navy', linestyle='--', label='Threshold=0.5')
axes[1].set_title("Fused Score Distribution")
axes[1].legend()

plt.tight_layout()
plt.show()


## Compare Fusion Strategies

In [ ]:
strategies = ["weighted_avg", "max", "vote"]
strat_results = []

for strat in strategies:
    ens = SentinelEnsemble(strategy=strat, threshold=0.5)
    f = ens.fuse(if_scores=if_scores, ae_scores=ae_scores, bert_scores=bert_scores)
    from sklearn.preprocessing import MinMaxScaler
    f = MinMaxScaler().fit_transform(f.reshape(-1,1)).flatten()
    strat_results.append({
        "Strategy": strat,
        "ROC-AUC": round(roc_auc_score(y_true, f), 4),
        "F1": round(f1_score(y_true, (f>=0.5).astype(int), zero_division=0), 4),
    })

print(pd.DataFrame(strat_results).to_string(index=False))


## Save Ensemble Config

In [ ]:
os.makedirs('ml/saved_models', exist_ok=True)
ensemble.save('ml/saved_models/ensemble_config.json')
print("Ensemble config saved.")
